# Construction Site Safety Detection — YOLOv8 Only

This notebook trains and evaluates **YOLOv8** (Ultralytics) on the Roboflow Construction Site Safety dataset in YOLO format.


In [ ]:
# (Colab) Mount Drive only if you load the zip from Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip your dataset (edit the path if needed)
# Example:
# !unzip "/content/drive/MyDrive/FinalProject/archive(9).zip" -d /content/FinalProject/

import os
DATA_DIR = "/content/FinalProject/css-data"
assert os.path.exists(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)
print("splits:", os.listdir(DATA_DIR)
)

In [ ]:
# Install YOLOv8 (Ultralytics)
!pip -q install ultralytics

from ultralytics import YOLO
import os, yaml, glob, random, cv2
import numpy as np

In [ ]:
# Class names (Roboflow Construction Site Safety)
CLASS_NAMES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'Machinery', 'Vehicle']
NUM_CLASSES = len(CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
# Create data.yaml for Ultralytics
DATA_DIR = "/content/FinalProject/css-data"
YOLO_DATA_YAML = "/content/data.yaml"

data_yaml = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)}
}
with open(YOLO_DATA_YAML, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("Saved:", YOLO_DATA_YAML)
print(open(YOLO_DATA_YAML, "r").read())

In [ ]:
# Quick dataset sanity check
def count_images(split):
    p = os.path.join(DATA_DIR, split, "images")
    return len(glob.glob(os.path.join(p, "*.*")))

def count_labels(split):
    p = os.path.join(DATA_DIR, split, "labels")
    return len(glob.glob(os.path.join(p, "*.txt")))

for s in ["train", "valid", "test"]:
    print(f"{s:5} -> images: {count_images(s)}, labels: {count_labels(s)}")

In [ ]:
# Train ONE model (recommended start: yolov8s)
# Tip: increase epochs for better results (e.g., 50–150)
model = YOLO("yolov8s.pt")
results = model.train(
    data=YOLO_DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,          # adjust if you hit out-of-memory
    project="/content/runs_yolo",
    name="yolov8s_css",
    patience=20
)

In [ ]:
# Validate on validation split (mAP, precision/recall, confusion matrix)
best_weights = "/content/runs_yolo/yolov8s_css/weights/best.pt"
model = YOLO(best_weights)

val_metrics = model.val(data=YOLO_DATA_YAML, split="val", imgsz=640, conf=0.25)
val_metrics

In [ ]:
# Test on test split (if you have labels in test)
test_metrics = model.val(data=YOLO_DATA_YAML, split="test", imgsz=640, conf=0.25)
test_metrics

In [ ]:
# Inference: visualize predictions on a few validation images
import matplotlib.pyplot as plt

VAL_IMG_DIR = os.path.join(DATA_DIR, "valid", "images")
imgs = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*.*")))
sample_imgs = random.sample(imgs, k=min(8, len(imgs)))

pred = model.predict(source=sample_imgs, imgsz=640, conf=0.25, save=False)

plt.figure(figsize=(14, 10))
for i, r in enumerate(pred):
    im = r.plot()  # BGR
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 4, i+1)
    plt.imshow(im)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Benchmark: speed (ms/img) and model size
import os, time

# model size
size_mb = os.path.getsize(best_weights) / (1024**2)
print(f"Best weights size: {size_mb:.2f} MB")

# simple FPS benchmark on N images
N = 100
imgs = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*.*")))
bench_imgs = imgs[:min(N, len(imgs))]

t0 = time.time()
_ = model.predict(source=bench_imgs, imgsz=640, conf=0.25, verbose=False)
t1 = time.time()

secs = t1 - t0
fps = len(bench_imgs) / secs if secs > 0 else float("inf")
print(f"Benchmark on {len(bench_imgs)} images: {fps:.2f} FPS (including preprocessing/postprocessing)")

In [ ]:
# Optional: Train multiple YOLO sizes (n/s/m/l/x) and compare quickly
# Warning: this will take longer. Uncomment to run.

# models = ["yolov8n.pt","yolov8s.pt","yolov8m.pt","yolov8l.pt","yolov8x.pt"]
# for w in models:
#     YOLO(w).train(
#         data=YOLO_DATA_YAML,
#         epochs=50,
#         imgsz=640,
#         project="/content/runs_yolo",
#         name=w.replace(".pt","") + "_css",
#         patience=20
#     )